In [7]:
!pip install -q openai-whisper
!pip install -q faster-whisper
!pip install -q transformers
!pip install -q datasets
!pip install -q jiwer
!pip install -q soundfile
!pip install -q psutil

In [8]:
import os
import time
import psutil
import soundfile as sf
import torchaudio
import whisper

from jiwer import wer, Compose, ToLowerCase, RemovePunctuation, Strip
from faster_whisper import WhisperModel
from transformers import pipeline

In [9]:
from datasets import load_dataset

dataset = load_dataset(
    "librispeech_asr",
    "clean",
    split="test[:3]"
)

print(dataset[0])

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

{'file': '/home/albert/.cache/huggingface/datasets/downloads/extracted/4a12a5d9e3d43248af424bf5fb40094f2aac834e9c19b34fb91aa912db1bd0b0/6930-75918-0000.flac', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7e0bebc6b6e0>, 'text': 'CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS', 'speaker_id': 6930, 'chapter_id': 75918, 'id': '6930-75918-0000'}


In [10]:
import soundfile as sf
import os

os.makedirs("audio_samples", exist_ok=True)

ground_truth = {}

for i, sample in enumerate(dataset):

    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    file_path = f"audio_samples/sample_{i}.wav"

    sf.write(file_path, audio, sr)

    ground_truth[file_path] = sample["text"]

print("Audio files saved successfully")

Audio files saved successfully


In [11]:
transformation = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    Strip()
])


def calculate_wer(predictions, references):

    normalized_predictions = [
        transformation(p)
        for p in predictions
    ]

    normalized_references = [
        transformation(r)
        for r in references
    ]

    error = wer(
        normalized_references,
        normalized_predictions
    )

    return error


WHISPER LARGE-V3 BENCHMARK

In [12]:
print("\n================================================")
print("RUNNING WHISPER LARGE-V3")
print("================================================")

whisper_model = whisper.load_model("small")

predictions_whisper = []
references_whisper = []

start_time = time.time()

for file_path in ground_truth.keys():

    result = whisper_model.transcribe(file_path)

    predictions_whisper.append(
        result["text"].strip()
    )

    references_whisper.append(
        ground_truth[file_path]
    )

end_time = time.time()

wer_whisper = calculate_wer(
    predictions_whisper,
    references_whisper
)

memory_whisper = (
    psutil.virtual_memory().used
    / (1024 ** 3)
)

time_whisper = end_time - start_time

print("\n===== Whisper Large-v3 Results =====")

for i in range(len(predictions_whisper)):

    print("\nPrediction:", predictions_whisper[i])
    print("Reference :", references_whisper[i])

print("\nWER:", wer_whisper)
print("Inference Time:", time_whisper)
print("Memory Usage (GB):", memory_whisper)



RUNNING WHISPER LARGE-V3

===== Whisper Large-v3 Results =====

Prediction: Concord returned to its place amidst the tents.
Reference : CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS

Prediction: The English voted to the French baskets of flowers of which they had made a plentiful provision to greet the arrival of the young princess. The French, in return, invited the English to a supper, which was to be given the next day.
Reference : THE ENGLISH FORWARDED TO THE FRENCH BASKETS OF FLOWERS OF WHICH THEY HAD MADE A PLENTIFUL PROVISION TO GREET THE ARRIVAL OF THE YOUNG PRINCESS THE FRENCH IN RETURN INVITED THE ENGLISH TO A SUPPER WHICH WAS TO BE GIVEN THE NEXT DAY

Prediction: Congratulations were poured in upon the princess everywhere during her journey.
Reference : CONGRATULATIONS WERE POURED IN UPON THE PRINCESS EVERYWHERE DURING HER JOURNEY

WER: 0.016129032258064516
Inference Time: 2.118568181991577
Memory Usage (GB): 2.27606201171875


FASTER-WHISPER BENCHMARK

In [13]:
from faster_whisper import WhisperModel
import time
import psutil

print("\n================================================")
print("RUNNING FASTER-WHISPER SMALL")
print("================================================")

faster_model = WhisperModel("small")

predictions_faster = []
references_faster = []

start_time = time.time()

for file_path in ground_truth.keys():

    segments, info = faster_model.transcribe(file_path)

    text = ""

    for segment in segments:
        text += segment.text

    predictions_faster.append(text.strip())
    references_faster.append(ground_truth[file_path])

end_time = time.time()

error = calculate_wer(
    predictions_faster,
    references_faster
)

memory_usage = (
    psutil.virtual_memory().used
    / (1024 ** 3)
)

print("\n===== Faster-Whisper Results =====")

for i in range(len(predictions_faster)):

    print("\nPrediction:", predictions_faster[i])
    print("Reference :", references_faster[i])

print("\nWER:", error)
print("Inference Time:", end_time - start_time)
print("Memory Usage (GB):", memory_usage)


RUNNING FASTER-WHISPER SMALL

===== Faster-Whisper Results =====

Prediction: Concord returned to its place amidst the tents.
Reference : CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS

Prediction: The English voted to the French baskets of flowers of which they had made a plentiful provision to greet the arrival of the young princess. The French, in return, invited the English to a supper, which was to be given the next day.
Reference : THE ENGLISH FORWARDED TO THE FRENCH BASKETS OF FLOWERS OF WHICH THEY HAD MADE A PLENTIFUL PROVISION TO GREET THE ARRIVAL OF THE YOUNG PRINCESS THE FRENCH IN RETURN INVITED THE ENGLISH TO A SUPPER WHICH WAS TO BE GIVEN THE NEXT DAY

Prediction: Congratulations were poured in upon the princess everywhere during her journey.
Reference : CONGRATULATIONS WERE POURED IN UPON THE PRINCESS EVERYWHERE DURING HER JOURNEY

WER: 0.016129032258064516
Inference Time: 1.9050192832946777
Memory Usage (GB): 2.715259552001953


DISTIL-WHISPER BENCHMARK

In [17]:
print("\n================================================")
print("RUNNING WHISPER TINY")
print("================================================")

tiny_model = whisper.load_model("tiny")

predictions_tiny = []
references_tiny = []

start_time = time.time()

for file_path in ground_truth.keys():

    result = tiny_model.transcribe(file_path)

    predictions_tiny.append(
        result["text"].strip()
    )

    references_tiny.append(
        ground_truth[file_path]
    )

end_time = time.time()

wer_tiny = calculate_wer(
    predictions_tiny,
    references_tiny
)

memory_tiny = (
    psutil.virtual_memory().used
    / (1024 ** 3)
)

time_tiny = end_time - start_time

print("\n===== Whisper Tiny Results =====")

for i in range(len(predictions_tiny)):

    print("\nPrediction:", predictions_tiny[i])
    print("Reference :", references_tiny[i])

print("\nWER:", wer_tiny)
print("Inference Time:", time_tiny)
print("Memory Usage (GB):", memory_tiny)


RUNNING WHISPER TINY


100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 269MiB/s]



===== Whisper Tiny Results =====

Prediction: Concord returned to its place amidst the tents.
Reference : CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS

Prediction: The English forwarded to the French baskets of flowers, of which they had made a plantable provision to greet the arrival of the young princess. The French in return invited the English to a supper, which was to be given the next day.
Reference : THE ENGLISH FORWARDED TO THE FRENCH BASKETS OF FLOWERS OF WHICH THEY HAD MADE A PLENTIFUL PROVISION TO GREET THE ARRIVAL OF THE YOUNG PRINCESS THE FRENCH IN RETURN INVITED THE ENGLISH TO A SUPPER WHICH WAS TO BE GIVEN THE NEXT DAY

Prediction: Congratulations, report in upon the Princess Everywhere, during her journey.
Reference : CONGRATULATIONS WERE POURED IN UPON THE PRINCESS EVERYWHERE DURING HER JOURNEY

WER: 0.04838709677419355
Inference Time: 0.9936261177062988
Memory Usage (GB): 7.868137359619141


In [19]:
import pandas as pd

results = {
    "Model": [
        "Whisper Small",
        "Faster-Whisper Small",
        "Whisper Tiny"
    ],

    "WER (%)": [
        round(wer_whisper * 100, 2),
        round(error * 100, 2),
        round(wer_tiny * 100, 2)
    ],

    "Inference Time (sec)": [
        round(time_whisper, 2),
        round(1.90, 2),   
        round(time_tiny, 2)
    ],

    "Memory Usage (GB)": [
        round(memory_whisper, 2),
        round(2.71, 2),   
        round(memory_tiny, 2)
    ],

    "Accuracy Level": [
        "Excellent",
        "Excellent",
        "Good"
    ],

    "Deployment Suitability": [
        "Moderate",
        "High",
        "Very High"
    ]
}

results_df = pd.DataFrame(results)

results_df.index = [
    "Model 1",
    "Model 2",
    "Model 3"
]

results_df

,Model,WER (%),Inference Time (sec),Memory Usage (GB),Accuracy Level,Deployment Suitability
Model 1,Whisper Small,1.61,2.12,2.28,Excellent,Moderate
Model 2,Faster-Whisper Small,1.61,1.90,2.71,Excellent,High
Model 3,Whisper Tiny,4.84,0.99,7.87,Good,Very High
